## IMPORT

In [207]:
import pandas as pd

raw_data = pd.read_csv('employee_attrition_2026_master.csv')
dictionary = pd.read_csv('data_dictionary.csv')

## Checking Missing / Duplicate / raw_data Type

In [208]:
raw_data.head()

,employee_id,age,generation,gender,department,job_level,is_manager,tenure_years,salary_usd,salary_vs_market_pct,...,burnout_score,manager_relationship_score,manager_1on1_per_month,engagement_score,promotions_last_2y,career_growth_score,recent_layoff_round,team_size,attrition_reason,attrition
0,EMP000000,26,Gen Z,Female,Marketing,Junior,0,5.5,70000,5.2,...,3.6,2,0,62,0,2,0,22,stayed,0
1,EMP000001,35,Millennial,Female,Sales,Junior,0,0.5,64000,-15.5,...,4.8,3,2,79,1,2,0,24,stayed,0
2,EMP000002,23,Gen Z,Male,HR,Senior,0,2.0,142000,-15.9,...,10.0,1,2,43,0,4,1,3,poor_management,1
3,EMP000003,20,Gen Z,Female,Sales,Manager,1,2.2,147000,2.6,...,4.3,1,4,64,1,3,0,24,stayed,0
4,EMP000004,40,Millennial,Female,Data,Senior,0,6.0,141000,11.3,...,2.9,1,1,92,0,2,0,5,stayed,0


In [209]:
raw_data.info()

print(f"duplicated = {raw_data.duplicated().sum()}")
print(f"size = {raw_data.size}")
print(f"shape = {raw_data.shape}\n")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45000 entries, 0 to 44999
Data columns (total 29 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   employee_id                 45000 non-null  object 
 1   age                         45000 non-null  int64  
 2   generation                  45000 non-null  object 
 3   gender                      45000 non-null  object 
 4   department                  45000 non-null  object 
 5   job_level                   45000 non-null  object 
 6   is_manager                  45000 non-null  int64  
 7   tenure_years                45000 non-null  float64
 8   salary_usd                  45000 non-null  int64  
 9   salary_vs_market_pct        45000 non-null  float64
 10  work_arrangement            45000 non-null  object 
 11  rto_mandate                 45000 non-null  int64  
 12  days_in_office_required     45000 non-null  int64  
 13  commute_minutes             450

## cleaning

|  | Column | Reason |
| :--- | :--- | :--- |
| 1 | employee_id | Unused identifier |
| 2 | attrition_reason | Not relevant for calculation |
| 3 | is_manager | Redundant with *job_level* |
| 4 | generation | Multicollinearity with *age* |


In [210]:
raw_data = raw_data.drop(columns=["employee_id", "attrition_reason", "is_manager","generation"])

### Delete data for individuals who started working before the age of 16.

In [211]:
bad_mask = raw_data["tenure_years"] > (raw_data["age"] - 16)
n_bad = bad_mask.sum()

clean_data = raw_data[~bad_mask].reset_index(drop=True)


In [212]:
clean_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43439 entries, 0 to 43438
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   age                         43439 non-null  int64  
 1   gender                      43439 non-null  object 
 2   department                  43439 non-null  object 
 3   job_level                   43439 non-null  object 
 4   tenure_years                43439 non-null  float64
 5   salary_usd                  43439 non-null  int64  
 6   salary_vs_market_pct        43439 non-null  float64
 7   work_arrangement            43439 non-null  object 
 8   rto_mandate                 43439 non-null  int64  
 9   days_in_office_required     43439 non-null  int64  
 10  commute_minutes             43439 non-null  int64  
 11  prefers_remote              43439 non-null  int64  
 12  flexibility_importance      43439 non-null  int64  
 13  ai_tools_adoption           434

## Outlier 

In [223]:
# 1. Check statistical outliers using IQR (continuous columns only)
cols = ["age", "tenure_years", "salary_usd", "commute_minutes", "weekly_hours"]

for col in cols:
    q1, q3 = clean_data[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outliers = ~clean_data[col].between(lo, hi)
    n_outliers = outliers.sum()
    print(f"{col:16s} bounds=({lo:.1f}, {hi:.1f}) outliers={n_outliers} max={clean_data[col].max()}")

# 2. Check logical conflict: remote workers required in office
bad_remote = (clean_data["work_arrangement"] == "remote") & (clean_data["days_in_office_required"] > 0)
print(f"\nRemote conflict rows: {bad_remote.sum()}")

# 3. Check salary logic by job level
print("\nSalary range by job level:")
print(clean_data.groupby("job_level")["salary_usd"].agg(["min", "max"]))

age              bounds=(12.5, 56.5) outliers=281 max=64
tenure_years     bounds=(-2.6, 8.9) outliers=1461 max=22.7
salary_usd       bounds=(-31500.0, 252500.0) outliers=650 max=345000
commute_minutes  bounds=(-29.5, 94.5) outliers=1426 max=180
weekly_hours     bounds=(23.0, 63.0) outliers=75 max=71

Remote conflict rows: 8574

Salary range by job level:
              min     max
job_level                
Director   118000  345000
Junior      35000   95000
Lead        80000  242000
Manager     88000  268000
Mid         44000  146000
Senior      60000  203000


In [228]:
# 1. Fix logical conflict: remote workers require 0 office days
clean_data.loc[clean_data["work_arrangement"] == "remote", "days_in_office_required"] = 0

# 2. Cap extreme outliers (1st to 99th percentile) without dropping rows
clip_cols = ["salary_usd", "tenure_years", "commute_minutes"]
for col in clip_cols:
    lo, hi = clean_data[col].quantile([0.01, 0.99])
    clean_data[col] = clean_data[col].clip(lo, hi)

print(f"Done, shape: {clean_data.shape}")

Done, shape: (43439, 25)


In [245]:
# 1. Verify remote conflicts resolved
conflict_mask = (clean_data["work_arrangement"] == "remote") & (clean_data["days_in_office_required"] > 0)
print(f"Remaining remote conflicts: {conflict_mask.sum()}")

# 2. Check min/max values after clipping
check_cols = ["salary_usd", "tenure_years", "commute_minutes"]
print(clean_data[check_cols].describe().loc[["min", "max"]])

Remaining remote conflicts: 0
       salary_usd  tenure_years  commute_minutes
min   47000.00000           0.3              3.0
max  263147.76336          11.4            118.0


# encoding 

job_level มีลำดับจริง (Junior<Mid<Senior<Lead<Manager<Director) OrdinalEncoder

ai_order มีลำดับจริง ("none<light<regular<power) OrdinalEncoder

gender, department, work_arrangement, ไม่มีลำดับ ใช้ OneHotEncoder


In [1]:
# OrdinalEncoder
job_level_order = ["Junior", "Mid", "Senior", "Lead", "Manager", "Director"]
clean_data["job_level"] = pd.Categorical(clean_data["job_level"], categories=job_level_order, ordered=True).codes
clean_data["job_level"].value_counts().sort_index()

NameError: name 'pd' is not defined

In [ ]:
# OrdinalEncoder
ai_order = ["none", "light", "regular", "power"]
clean_data["ai_tools_adoption"] = pd.Categorical(clean_data["ai_tools_adoption"], categories=ai_order, ordered=True).codes
clean_data["ai_tools_adoption"].value_counts().sort_index()

In [ ]:
# OneHotEncoder
nominal_cols = ["gender", "department", "work_arrangement"]
clean_data = pd.get_dummies(clean_data, columns=nominal_cols ,drop_first=True)

In [ ]:
clean_data

# สรุป
มีข้อมูล ทั้งหมด 43439 rows × 37 columns


